# Evaluation Metrics for Generative Modeling on Spatial Transcriptomics
**Data**: paired spatial-transcriptomics slides (e.g. Mouse Brain Atlas / ABCA)

A runnable companion to the `nicheflow_eval` metric suite. Each section below explains one metric
and shows the **exact code to run it on its own**, against a real **target slide** and a set of
**generated cells**. The metrics evaluate how a generative model (e.g. the flow-matching models in
NicheFlow) reproduces gene expression **and** spatial coordinates simultaneously.

Run **§0 Environment setup** and **§0b Data loading** first; after that every metric section is
independent and can be run on its own. Inputs are plain arrays — no flow model, checkpoint, or
Hydra:

- `TargetSlide` — the real target slide: `x (N,P)` PCA expression, `pos (N,2)` coords, `ct (N,)`
  cell types, optional grid subsample.
- `GeneratedNiches` — `(B, N, D)` generated microenvironments, **centroid at point 0**.

## 0. Environment setup

Install the package (editable), then check which optional backends are present — some metrics need
extra deps: `torch`+`pot` for EMD/MMD, `squidpy` for Moran's I.

```bash
cd nicheflow-eval
uv venv && uv pip install -e .       # or:  pip install -e .
```

In [ ]:
import importlib
import numpy as np
import pandas as pd


def _has(mod):
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False


print("optional backends present:")
print("  torch   (MMD / EMD / classifier):", _has("torch"))
print("  ot/pot  (EMD / Wasserstein)     :", _has("ot"))
print("  squidpy (Moran's I)             :", _has("squidpy"))

## 0b. Data loading

This notebook assumes the `data/` folder already contains:

- **`target_abca.pkl`** — the **target slide on its own** (its own preprocessing dataclass). We
  build `TargetSlide` from this. It carries `X_pca` (the **shared PCA space** the generated cells live
  in), `coords`, cell types, and the grid subsample — so no scanpy re-run is needed.
- **`abca_aligned_clf.pkl`** — the **held-out same-mouse slide** the *neutral* concordance
  classifier is trained on (see §6). Not loaded here; it is the classifier's training data.
- **`generated.npz`** — the generated results: arrays `x (B,N,P)`, `pos (B,N,P)`, and the paired
  real target niches `gt_x`, `gt_pos` (needed by the regression and concordance metrics).

`target_abca.pkl` and `generated.npz` are **required** — if either is missing the next cell raises
`FileNotFoundError` (there is **no** synthetic fallback). `gt_x`/`gt_pos` inside the `.npz` are
optional: without them the regression and concordance sections are skipped.

In [ ]:
import os

from nicheflow_eval import TargetSlide, GeneratedNiches
from nicheflow_eval.data import load_h5ad_dataset_dataclass

TARGET_PKL = "../data/target_abca.pkl"     # the TARGET slide on its own (not the aligned src+tgt pkl)
GENERATED_PATH = "../data/generated.npz"   # arrays: x (B,N,P), pos (B,N,P); optional gt_x, gt_pos
SEED = 0

# Required inputs — no synthetic fallback. Fail loudly if either is missing.
if not os.path.exists(TARGET_PKL):
    raise FileNotFoundError(
        f"Target slide pkl not found: {TARGET_PKL}. Set TARGET_PKL to the target slide's own "
        f"preprocessing .pkl (not the aligned source+target pkl)."
    )
if not os.path.exists(GENERATED_PATH):
    raise FileNotFoundError(
        f"Generated cells .npz not found: {GENERATED_PATH}. Set GENERATED_PATH to an .npz with "
        f"arrays x (B,N,P), pos (B,N,P), and (for regression + concordance) gt_x, gt_pos."
    )

# --- target slide (its own pkl) ---
ds = load_h5ad_dataset_dataclass(TARGET_PKL)
timepoint = ds.timepoints_ordered[-1]                 # target_abca.pkl holds just the target slide
target = TargetSlide.from_dataclass(ds, timepoint=timepoint)
print(f"target '{timepoint}': {target.x.shape[0]} cells, {target.x.shape[1]} PCs, "
      f"{target.n_classes} cell types")

# --- generated cells ---
npz = np.load(GENERATED_PATH)
extra = {k: npz[k] for k in ("gt_x", "gt_pos") if k in npz}
generated = GeneratedNiches(x=npz["x"], pos=npz["pos"], **extra)

print(f"generated: {generated.x.shape[0]} niches x {generated.x.shape[1]} points "
      f"= {generated.flat_x.shape[0]} cells")
print(f"flat cloud: x{generated.flat_x.shape}  pos{generated.flat_pos.shape}")
if generated.gt_x is None or generated.gt_pos is None:
    print("note: no gt_x/gt_pos in the .npz -> regression and concordance will be skipped.")

## 1. Pointwise regression (MSE / MAE) — original NicheFlow

The most basic metric: per matched cell, the error between the final generated state and its
ground-truth target cell, scored separately for gene expression (`x`) and coordinates (`pos`).
Lower = better. Computed by `regression_metrics` in `nicheflow_eval.metrics.distances`.

- `test/x/mse` — **mean squared error** of predicted vs. true **gene expression** (lower = better).
- `test/x/mae` — **mean absolute error** of predicted vs. true **gene expression** (lower = better).
- `test/pos/mse` — MSE of predicted vs. true **spatial coordinates** (lower = better).
- `test/pos/mae` — MAE of predicted vs. true **coordinates** (lower = better).

These reward *exact per-cell* prediction (complementary to the distribution-level EMD/MMD below).
Needs the matched ground-truth microenvironment per generated niche (`generated.gt_x/gt_pos`).
Note NicheFlow reports MSE/MAE but **not** R².

In [ ]:
from nicheflow_eval.metrics.distances import regression_metrics

if generated.gt_x is not None and generated.gt_pos is not None:
    res = regression_metrics(generated.x, generated.gt_x, generated.pos, generated.gt_pos, prefix="test")
    for k, v in res.items():
        print(f"{k:14s} {v:.4f}")
else:
    print("Regression needs matched ground truth: set generated.gt_x / generated.gt_pos "
          "(the true target microenvironment per generated niche).")

## 2. EMD

Earth-mover (Wasserstein) distance between the **pooled generated cells** and **all real target
cells**, computed **separately** for gene expression (`x`, 50-d PCA) and spatial coordinates
(`pos`, 2-d) — they live on different scales, so they are scored independently. Lower = the
generated distribution is closer to the real one. Computed with exact OT (POT `emd2`) in
`nicheflow_eval.metrics.distribution.ot_distance`.

- `test/ot_w1/x` — **Wasserstein-1** distance on **gene expression** (lower = better).
- `test/ot_w1/pos` — Wasserstein-1 on **coordinates** (lower = better).
- `test/ot_w2/x` — **Wasserstein-2** on **expression** (penalizes large transport more than W1; lower = better). *Checkpoint-selection metric (`val/ot_w2/x`).*
- `test/ot_w2/pos` — Wasserstein-2 on **coordinates** (lower = better).

Caveat: scoring `x` and `pos` separately cannot detect a wrong expression↔position **coupling** —
that is what the C2ST (§8) adds.

In [ ]:
from nicheflow_eval.metrics.distribution import ot_distance

# pooled generated cloud vs all real target cells, scored separately for x and pos
real_x, real_pos = target.x, target.pos
gen_x, gen_pos = generated.flat_x, generated.flat_pos

print("test/ot_w1/x  ", round(ot_distance(real_x,   gen_x,   power=1, seed=SEED), 4))
print("test/ot_w1/pos", round(ot_distance(real_pos, gen_pos, power=1, seed=SEED), 4))
print("test/ot_w2/x  ", round(ot_distance(real_x,   gen_x,   power=2, seed=SEED), 4))
print("test/ot_w2/pos", round(ot_distance(real_pos, gen_pos, power=2, seed=SEED), 4))

## 3. MMD

Maximum Mean Discrepancy (squared, RBF kernel) between the pooled generated cells and all real
target cells — a kernel two-sample distance — again scored **separately** for expression and
coordinates. Lower = distributions more similar. Computed with
`nicheflow_eval.metrics.distribution.mmd2_rbf`.

- `test/mmd2/x` — squared MMD on **gene expression** (lower = better).
- `test/mmd2/pos` — squared MMD on **coordinates** (lower = better).

Like EMD, it compares the `x` and `pos` **marginals** independently. (The unbiased U-statistic can
dip slightly negative when the distributions match — that is expected, do not clip.)

In [ ]:
from nicheflow_eval.metrics.distribution import mmd2_rbf

print("test/mmd2/x  ", round(mmd2_rbf(target.x,   generated.flat_x,   seed=SEED), 5))
print("test/mmd2/pos", round(mmd2_rbf(target.pos, generated.flat_pos, seed=SEED), 5))

## 4. Point-to-Shape distance

For each generated cell, distance to its nearest real target cell.

- `test/psd/mean` — average nearest-real-neighbour distance of generated cells, i.e. how close generations land to the real manifold (lower = better).
- `test/psd/max` — worst-case (largest) nearest-neighbour distance, capturing the most off-manifold generated cell (lower = better).

In [ ]:
from nicheflow_eval.metrics.distances import point_to_shape

res = point_to_shape(generated.flat_pos, target.pos, prefix="test")
for k, v in res.items():
    print(f"{k:14s} {v:.4f}")

## 5. Shape-to-Point distance

The reverse direction: for each real target cell, distance to its nearest generated cell (a coverage measure).

- `test/spd/mean` — average distance from real cells to the nearest generated cell, i.e. how well generations **cover** the real distribution (lower = better).
- `test/spd/max` — largest such distance, flagging the most poorly covered real region (lower = better).

In [ ]:
from nicheflow_eval.metrics.distances import shape_to_point

res = shape_to_point(generated.flat_pos, target.pos, prefix="test")
for k, v in res.items():
    print(f"{k:14s} {v:.4f}")

## 6. Cell-type Classifier-based metric (neutral held-out classifier)

A **neutral** SetTransformer spatial classifier — trained on a **held-out same-mouse slide**
(`abca_aligned_clf.pkl`, *neither* source nor target, projected into the source+target PCA basis +
label space) — labels **both** the generated niche and its **paired real target niche** (`gt_*`,
same centroid). Because the same annotator scores both sides and it never saw the target slide,
this avoids the leakage/circularity of the old nearest-real-cell pseudo-label.
`cell_type_concordance` (`nicheflow_eval.metrics.concordance`) reports:

- `test/ct/f1` — weighted multiclass **F1** of the classifier's labels on generated vs. real niches (higher = better).
- `test/ct/acc` — **accuracy** of the same agreement (higher = better).
- `test/ct/prop_kl` — **KL divergence** between the generated/real **label-proportion** histograms (lower = better).
- `test/ct/prop_tv` — **total-variation** distance between those proportions (lower = better).
- `test/ct/prop_jsd` — **Jensen-Shannon** divergence between those proportions (symmetric; lower = better).

**f1 / acc** = per-niche agreement (is each generated niche labelled like the real one?);
**prop_kl / prop_tv / prop_jsd** = population-level cell-type composition fidelity.

> Needs (a) the paired real niches `generated.gt_x / gt_pos` and (b) a classifier trained on the
> held-out slide:
> `python -m nicheflow_eval.classifier.train experiment=classifier/abca_spatial`
> (its `data.datamodule.data_fp` points at `abca_aligned_clf.pkl`). Point `CLASSIFIER_CKPT` at the
> resulting checkpoint. Loading via `load_spatial_classifier` also carries the training
> `n_neighbors` onto the net, so eval builds identically sized niches.

In [ ]:
import torch

from nicheflow_eval.classifier.nets import SpatialCTClassifierNet
from nicheflow_eval.metrics._common import load_spatial_classifier
from nicheflow_eval.metrics.concordance import cell_type_concordance

CLASSIFIER_CKPT = "../ckpts/Classifier_Spatial_ABCA_SetTransformer.ckpt"

clf = SpatialCTClassifierNet(
    input_dim=target.x.shape[1],
    output_dim=target.n_classes,
    hidden_dim=64,
    coord_dim=2,
    num_heads=4,
    mask_centroid=True,
)

if os.path.exists(CLASSIFIER_CKPT):
    load_spatial_classifier(clf, torch.load(CLASSIFIER_CKPT, map_location="cpu"))  # +n_neighbors
    print("loaded trained neutral classifier")
else:
    print("no checkpoint -> UNTRAINED classifier (numbers are meaningless; "
          "train one on abca_aligned_clf.pkl first).")

if generated.gt_x is not None and generated.gt_pos is not None:
    res = cell_type_concordance(
        generated.x, generated.pos, generated.gt_x, generated.gt_pos, clf,
        prefix="test", spatial=True, n_classes=target.n_classes,   # n_neighbors comes from the ckpt
    )
    for k, v in res.items():
        print(f"{k:18s} {v:.4f}")
else:
    print("concordance needs the paired real niches generated.gt_x / generated.gt_pos.")

## 7. Moran's I Score

Moran's I measures whether a feature is spatially **clustered** (nearby cells have similar values).
In the PC space (e.g. `n_pcs = 50`), `morans_compare` (`nicheflow_eval.metrics.morans`) computes
it **per PC** on the generated grid centroids and on the paired real grid subsample of the target
slide (same squidpy kNN graph, `n_neighs=6`), then compares the two per-PC vectors,
**variance-weighted** by real per-PC variance.

| Metric | What it compares | Reference (ABCA) |
|---|---|---|
| `moran/corr` ↑ | weighted Pearson corr of per-PC `I_real` vs `I_gen` across PCs | 0.952 (0.948–0.954) |
| `moran/mae` ↓ | weighted mean `\|I_real − I_gen\|` over PCs | 0.039 (0.037–0.041) |
| `moran/real_mean` | weighted mean Moran's I of the **real** slide (diagnostic) | 0.0665 |
| `moran/gen_mean` | weighted mean Moran's I of the **generated** slide (diagnostic) | 0.081 |

### What each Moran metric means
- **`moran/corr`** — across the PCs, do the PCs that are spatially clustered in the real slide come
  out clustered in the generated slide too? 1.0 = the **relative** clustering pattern is perfectly
  reproduced; 0 = unrelated.
- **`moran/mae`** — the average absolute gap between real and generated Moran's I per PC (after
  variance weighting). 0 = identical clustering strength PC-by-PC.
- **`moran/real_mean` / `moran/gen_mean`** — diagnostics, not a score: the overall clustering level
  of each slide. Comparing them tells you the *direction* of any mismatch (over- vs under-clustered).

In [ ]:
from nicheflow_eval.metrics.morans import morans_compare

# generated grid centroids (point 0 of each niche) vs the matched real grid subsample
grid_x, grid_pos = target.moran_grid   # the .pkl grid subsample if present, else the full cloud

res = morans_compare(
    generated.centroid_x, generated.centroid_pos, grid_x, grid_pos,
    prefix="test", n_neighs=6, seed=SEED,
)
for k, v in res.items():
    print(f"{k:18s} {v:.4f}")

## 8. Classifier Two-Sample Test

C2ST trains a classifier to tell **real** target cells from **generated** ones; **no cell-type
labels or pseudo-labels** are used, so it sidesteps the caveat of the concordance metrics (§6).
Each score is a held-out accuracy/AUC: **~0.5 = indistinguishable (good), ~1.0 = trivially
separable (bad)**. Computed by `c2st_metrics` (`nicheflow_eval.metrics.c2st`).

| Metric | What it compares | Reference (ABCA) |
|---|---|---|
| `c2st/acc` | per-cell **joint `[expression \| position]`** | 0.598 (0.590–0.608) |
| `c2st/auc` | "" (ROC-AUC) | 0.633 (0.626–0.636) |
| `c2st/pos_acc` | per-cell **position only** (diagnostic) | 0.561 (0.549–0.578) |

### What each C2ST metric means
- **`c2st/acc`** — accuracy of an MLP distinguishing real vs generated using the **joint**
  `[expression | position]` vector: the **co-generation** test (detects a wrong expression↔position
  coupling the separate MMD/EMD scores cannot). 0.5 = indistinguishable.
- **`c2st/auc`** — the same test by ROC-AUC (threshold-independent; robust to class imbalance).
- **`c2st/pos_acc`** — diagnostic C2ST on **coordinates only**; surfaces spatial problems that the
  ~50-d joint vector might dilute. 0.5 = the generated spatial *marginal* matches real.

### Significance test (label permutation)
Set `n_perm > 0` (e.g. 100) to test whether the joint AUC is **significantly above chance**: pool
real + generated, shuffle the labels `n_perm`×, recompute the AUC to build a null. Adds
`c2st/sig_auc`, `c2st/sig_null_p95`, `c2st/sig_pval` — **significant if `sig_auc > sig_null_p95`
(pval < 0.05)**.

In [ ]:
from nicheflow_eval.metrics.c2st import c2st_metrics

res = c2st_metrics(
    target.x, target.pos,
    generated.flat_x, generated.flat_pos,
    prefix="test",
    n_folds=5,
    n_perm=0,                            # set to 100 to add the label-permutation significance test
    seed=SEED,
)
for k, v in res.items():
    print(f"{k:20s} {v:.4f}")

## 9. Run the whole suite at once

`evaluate` runs every applicable group and returns a flat `{test/group/metric: value}` dict (the
columns the NicheFlow result CSVs use). Groups whose inputs are missing are listed under
`_skipped`. Add `"regression"` once `generated.gt_*` is set and `"concordance"` once you pass a
trained `classifier=`.

In [ ]:
from nicheflow_eval import evaluate

groups = ("psd", "spd", "distribution", "c2st", "moran")
all_res = evaluate(target, generated, groups=groups, seed=SEED)
print("skipped:", all_res.pop("_skipped"))
pd.DataFrame(sorted(all_res.items()), columns=["metric", "value"])